# Day 2 - Preprocessing and Baseline Models

We clean and encode the data (using the functions you wrote in `src/preprocessing.py`), split it into train/test, and train two baseline classifiers. The goal is a first honest number to beat on Day 3.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from src.data_loader import load_raw_data
from src.preprocessing import prepare_data, split_data

df = load_raw_data()
X, y = prepare_data(df)
X_train, X_test, y_train, y_test = split_data(X, y)

print("feature matrix shape:", X.shape)
print("train / test rows   :", X_train.shape[0], "/", X_test.shape[0])
print("churn rate  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

feature matrix shape: (7043, 30)
train / test rows   : 5634 / 1409
churn rate  train: 0.265  test: 0.265


## Baseline 1 - Logistic Regression

Linear models are sensitive to feature scale (tenure is 0-72, MonthlyCharges is ~18-119), so we put a `StandardScaler` in front using a `Pipeline`. The tree model below does not need this.

In [2]:
logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
logreg.fit(X_train, y_train)

lr_pred = logreg.predict(X_test)
lr_proba = logreg.predict_proba(X_test)[:, 1]
print("accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, lr_proba), 4))
print(classification_report(y_test, lr_pred))

accuracy: 0.807
ROC-AUC : 0.8418
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## Baseline 2 - Random Forest

In [3]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
print("accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, rf_proba), 4))
print(classification_report(y_test, rf_pred))

accuracy: 0.7928
ROC-AUC : 0.8264
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.64      0.51      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.78      1409



## Your reflection (write your answers here)

Look at the two reports above, then answer in one or two sentences each:

1. Which model has the better **ROC-AUC**?

   The LogReg(0.8418) model is giving better ROC-AUC than RandomForest(0.8264).
3. A lazy model that always predicts "No churn" would already be right about **73%** of the time. Given that, why is the **recall of the churn class (label `1`)** more useful to us than overall accuracy?
   
   Because there can business loss like what is we missed a churner, he silently walk away, we didn't notice, revenue lost.The second one, what if we give discount to someone who would have stayed anyway, that's also a mistake from our side.
   That's why recall of churn class is more useful, try to make them stay more rather than focusing on people who would leave anyway, why lose revenue on the these customers.

This is the exact intuition Day 3's tuning is built on.